# `ask_anything()` — the capability typology as one executable function

One field, any question. The system decides the bucket and answers **only in the way that
bucket can be answered faithfully**:

```
ask_anything(question)
 |- router: aggregate form?  no  -> BUCKET 1  retrieval -> grounded answer + citations
 `- yes -> which column does the answer need?
     |- validated CONTENT column (alienation_alleged)  -> BUCKET 3 (extracted):
     |       threshold-aware count + abstention audit
     |- METADATA columns                                -> BUCKET 2: NL->SQL with guardrails
     |       (coder model, few-shot, temp 0, EXPLAIN-validated, SQL printed for review)
     `- no column carries the answer                    -> BUCKET 3 (unextracted): refusal
```

Nothing here is new machinery: the router, retrieval, DuckDB layer, threshold helper **and
the NL->SQL translator** are exec'd from `rag_echr_ris.ipynb` and `echr_query.ipynb`.
The metadata branch is genuinely semantic — the local coder model maps "against Poland" to
`respondent_state = 'POL'` and "Kanton Bern" to the Swiss table itself — behind **two
deterministic refusal nets**: a deny-list of known-unextracted content concepts (semantic
knowledge no validator can derive) and `EXPLAIN` schema validation of every generated
statement. Refusal is never delegated to the model — measured: a 3B model offered a
CANNOT_ANSWER escape takes it whenever the question needs composition. The generated SQL
always prints **above** its result: for aggregate answers the trust boundary is the query
text, not the number. The citable quantitative path remains the reviewed canned queries.

## 1. Load the deployed pipeline (RAG cells + query-layer cells, cache-hot)

In [ ]:
import json
from pathlib import Path

RAG_NB   = Path("rag_echr_ris.ipynb")
QUERY_NB = Path("echr_query.ipynb")

def exec_cells(nb_path, markers):
    nb = json.loads(nb_path.read_text())
    n = 0
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        src = "".join(c["source"])
        if any(m in src for m in markers):
            exec(src, globals())
            n += 1
    print(f"exec'd {n} cells from {nb_path.name}")

# RAG pipeline: config, loaders, chunker, genre, corpus, embedder/index, retrieve,
# router (Layer 1), generation + answer()
exec_cells(RAG_NB, ["DATA_DIR  = Path", "def load_json_records", "ECHR_ANCHORS = [",
                    "reusable ECHR genre helper", "records, chunks = [], []",
                    "_embedder = None", "def _to_hit", "AGGREGATE_PATTERNS = [",
                    "SYSTEM_PROMPT = ("])
assert index is not None, "index missing -- run rag_echr_ris.ipynb once first"

# Query layer: config, table -> DuckDB, threshold-aware helper, NL->SQL translator
exec_cells(QUERY_NB, ["MIN_CONFIDENCE = 0.70", "def _load_table", "def alienation_at(",
                      "def nl2sql"])
assert con is not None, "DuckDB table missing -- run echr_extraction + theme classify first"

print(f"ready: {index.ntotal} chunks | {len(df)} table rows | router: {len(AGGREGATE_PATTERNS)} patterns")

inputs: ['echr_parental_alienation.json', 'ris_parental_alienation.json', 'swiss_parental_alienation.json']
chunk=600w/80o | ECHR skip={'OPINION', 'RELEVANT_LAW'} law_only=False | dev_cap=None
RIS decisions=True (civil only) | Swiss=True
genre routing: exclude={'communicated'} | MMR fetch_k=30 lambda=0.7
normalisers ready: ['echr', 'ris', 'swiss']
section splitter + chunker ready
genre helper ready: ('merits', 'admissibility', 'communicated', 'other') | low-info: {'communicated'}
  echr : 1116 records from echr_parental_alienation.json
  ris : 38 principles + 479 civil decisions (dropped 31 criminal-senate/AUSL 'Text' records — Entfremdung homonym / ECtHR summaries)
  ris  : 517 records from ris_parental_alienation.json
  swiss: dropped 24 Rechenschaftsbericht records (court annual reports, not case law — multi-case digests that flood the top-k)
  swiss: 2007 records from swiss_parental_alienation.json

ECHR dates: 1114/1116 have YYYY-MM-DD, 2 n.d.
ECHR sections: 873/1116 parsed; 243 f

## 2. Metadata relations for the Bucket-2 handlers
Quiet re-registration of the cross-jurisdiction metadata (same fields as `echr_query.ipynb`
section 6b; the coverage audit lives there).

In [ ]:
import json as _json
import re as _re
import pandas as pd

_swiss = _json.loads((DATA_DIR / "swiss_parental_alienation.json").read_text())
_ris   = _json.loads((DATA_DIR / "ris_parental_alienation.json").read_text())

swiss_meta = pd.DataFrame([{"id": r.get("stable_id"), "canton": r.get("canton"),
                            "court_type": r.get("court_type"), "year": r.get("year")}
                           for r in _swiss])

_SEN = _re.compile(r"\d{1,3}\s*([A-Z][a-z]{1,2})")

def _senate(gz):
    m = _SEN.match((gz or "").split(";")[0].strip())
    return m.group(1) if m else None

ris_meta = pd.DataFrame([{
    "id": r.get("id"), "dokumenttyp": r.get("dokumenttyp"),
    "year": int(r["entscheidungsdatum"][:4]) if (r.get("entscheidungsdatum") or "")[:4].isdigit() else None,
    "senate": _senate(r.get("geschaeftszahl")),
} for r in _ris])

# ECHR metadata incl. the 2026-07-07 fields (formation, introduction->judgment duration)
from datetime import datetime as _dt
_echr_raw = _json.loads((DATA_DIR / "echr_parental_alienation.json").read_text())

def _pdate(s):
    s = (s or "").split()[0] if s else ""
    try:
        return _dt.strptime(s, "%d/%m/%Y").date()
    except (ValueError, IndexError):
        return None

def _formation(dc):
    for f in ("GRANDCHAMBER", "CHAMBER", "COMMITTEE"):
        if f in (dc or ""):
            return f
    return None

def _dur(r):
    if (r.get("doctype") or "").upper() == "HECOM":
        return None                      # communicated = still pending, no duration
    a = _pdate(r.get("introductiondate"))
    b = _pdate(r.get("judgementdate")) or _pdate(r.get("decisiondate"))
    return (b - a).days if a and b and b >= a else None

echr_meta = pd.DataFrame([{
    "id": r.get("itemid"), "respondent": r.get("respondent"),
    "importance": int(r["importance"]) if str(r.get("importance", "")).isdigit() else None,
    "year": int(r["ecli"].split(":")[3][:4]) if (r.get("ecli") or "").startswith("ECLI:CE:ECHR:") else None,
    "formation": _formation(r.get("documentcollectionid")),
    "duration_days": _dur(r),
    "separate_opinion": str(r.get("separateopinion", "")).upper() == "TRUE",
} for r in _echr_raw])
echr_kp = pd.DataFrame([
    {"id": r.get("itemid"), "kp_code": code.strip()}
    for r in _echr_raw for code in (r.get("kpthesaurus") or "").split(";") if code.strip()
])

con.register("swiss_meta", swiss_meta)
con.register("ris_meta", ris_meta)
con.register("echr_meta", echr_meta)
con.register("echr_kp", echr_kp)

# make the German corpora visible to the NL->SQL translator
SCHEMA = SCHEMA + (
    " Additional tables: swiss_meta(id TEXT, canton TEXT two-letter Swiss canton e.g. "
    "ZH=Zuerich, BE=Bern, AG=Aargau, BL=Basel-Land, BS=Basel-Stadt, GR=Graubuenden, "
    "SG=St.Gallen, CH=federal, court_type TEXT, year INT) = Swiss decisions; "
    "ris_meta(id TEXT, dokumenttyp TEXT[Rechtssatz|Text], year INT, senate TEXT) "
    "= Austrian OGH records.")
FEW_SHOT = FEW_SHOT + [
    ("Aus welchem Kanton stammen die meisten Schweizer Entscheidungen?",
     "SELECT canton, COUNT(*) AS n FROM swiss_meta GROUP BY canton ORDER BY n DESC LIMIT 5;"),
]
print(f"registered swiss_meta ({len(swiss_meta)}) + ris_meta ({len(ris_meta)}) + "
      f"echr_meta ({len(echr_meta)}, duration coverage "
      f"{echr_meta.duration_days.notna().mean()*100:.0f}%) + echr_kp ({len(echr_kp)}) | "
      f"schema + few-shot extended")

registered swiss_meta (2031) + ris_meta (548) + echr_meta (1116, duration coverage 28%) + echr_kp (5345) | schema + few-shot extended


## 3. The dispatcher — content column, deny-list, then guarded NL→SQL
Order matters and each step is the *cheapest sufficient* mechanism:

1. **Validated content column** (`alienation_alleged`) — keyword-routed, because it must reach
   the calibrated-confidence machinery, and there is exactly one such column.
2. **Deny-list of known-unextracted concepts** (custody outcome, marital status, duration) —
   *semantic* knowledge about what is NOT in the data; no SQL validator can know it, and
   (measured) a 3B model given a refusal escape over-uses it on composable questions, so the
   model is never asked to refuse.
3. **Guarded NL→SQL** for everything else — the local coder model maps entities itself
   ("against Poland" → `respondent_state='POL'`, "Kanton Bern" → `swiss_meta`, canton `BE`),
   behind `EXPLAIN` validation with one retry. The generated SQL prints above its result —
   review the query, not the number.

Failure direction of every net is **refusal, never fabrication**.

In [ ]:
def _h_alienation(q):
    """BUCKET 3 (extracted): the one validated content column, threshold-aware."""
    res, audit = alienation_at(min_confidence=MIN_CONFIDENCE, mode="filter")
    print("  BUCKET 3 (content aggregate -- extracted + calibrated)")
    print(f"  ANSWER: {audit['passed']} cases with a confident alienation allegation "
          f"(calibrated conf >= {audit['min_confidence']}); "
          f"{audit['abstained_low_conf']} further predicted-positive cells ABSTAINED "
          f"(low confidence, surfaced not dropped); basis={audit['confidence_basis']}")
    print(res[["id", "title", "respondent_state", "conf_eff"]].head(5).to_string(index=False))
    print("  CAVEAT: extractor validated at F1 0.51-0.59 on 120 gold labels; ECHR Article-8 "
          "English cases only -- a calibrated estimate, not a hard fact.")


REFUSAL = ("BUCKET 3 (content aggregate -- NOT extracted): no validated per-case column "
           "carries this answer (e.g. custody outcome, marital status). "
           "Answering from retrieval would fabricate a statistic; the faithful options are "
           "building + validating an extractor for that field (see echr_extraction.ipynb) "
           "or this refusal.")

# net 1 -- known-unextracted content concepts (semantic knowledge SQL validation cannot have):
_UNEXTRACTED = _re.compile(
    r"(receiv\w*|erhielt|erh\u00e4lt|awarded|granted|zugesprochen)\s+(the\s+)?(sole\s+)?"
    r"(custody|sorgerecht|obhut)"
    r"|(custody|sorgerecht|obhut)\s+(was\s+)?(receiv|award|grant|zugesprochen)"
    r"|\bmarried\b|\bverheiratet\b",
    _re.IGNORECASE)   # duration removed 2026-07-07: introductiondate arrived,
                      # proceedings duration moved from refusal to Bucket 2


def _h_nl2sql(q):
    """BUCKET 2 (metadata aggregate) via guarded NL->SQL. Two deterministic nets:
    deny-list (above, checked inside nl2sql too) and EXPLAIN validation + retry.
    Refusal is never delegated to the model (measured lazy-escape effect)."""
    sql = nl2sql(q)
    if sql is None:
        print("  BUCKET 2 (metadata aggregate) -- UNAVAILABLE, not refused:")
        print("  the local NL->SQL translator is unreachable (start `ollama serve` and re-run).")
        print("  The question IS answerable from metadata; see the canned queries in echr_query.ipynb.")
        return
    if sql.startswith("--"):
        print(f"  NL->SQL declined/invalid: {sql}")
        print(" ", REFUSAL); return
    print("  BUCKET 2 (metadata aggregate) -- generated SQL (REVIEW THIS, it is the trust boundary):")
    print("   ", sql)
    try:
        print(run_sql(sql).head(12).to_string(index=False))
    except Exception as e:
        print("  execution failed:", e); print(" ", REFUSAL); return
    print("  CAVEAT: counts describe the keyword-matched corpora, never litigation rates; "
          "citable numbers come from the reviewed canned queries in echr_query.ipynb.")


def ask_anything(question, k=4):
    print("=" * 88)
    print("Q:", question)
    trig = aggregate_trigger(question)
    if not trig:
        print("ROUTE: Bucket 1 (no aggregate trigger) -> retrieval + grounded generation")
        r = answer(question, k=k)
        if r["abstained"]:
            print(f"  abstained: {r['abstain_type']} -- {r['answer'][:200]}")
        elif r["answer"]:
            print("  ANSWER:", r["answer"][:600].replace("\n", " "))
        else:
            print("  (generation OFF -- LLM unreachable; retrieval-only mode, hits below)")
        for n, h in enumerate(r["hits"][:3], 1):
            print(f"  [{n}] cos={h['score']:.3f} | {h['jurisdiction']:9s} | {h['title'][:55]}")
        return
    print(f"ROUTE: aggregate trigger '{trig}' -> query layer (never generation)")
    if _re.search(r"alienat|entfremd", question, _re.IGNORECASE):
        _h_alienation(question)                      # validated content column first
    elif _UNEXTRACTED.search(question):
        print(" ", REFUSAL)                          # known-unextracted concept
    else:
        _h_nl2sql(question)                          # semantic metadata path


print("ask_anything() ready -- dispatch: content column -> deny-list -> guarded NL->SQL")

ask_anything() ready -- dispatch: content column -> deny-list -> guarded NL->SQL


## 4. One entry point, seven questions, four distinct behaviours
Two Bucket-1 (answered with citations, one DE one EN), one aggregate in each metadata
dimension, the extracted content aggregate with its abstention audit, and one question the
system **correctly refuses**.

In [ ]:
DEMO = [
    "When can custody be transferred to the other parent because of alienating behaviour?",
    "Unter welchen Voraussetzungen kann einem Elternteil die Obhut entzogen werden?",
    "How many cases against Poland are in the corpus?",              # NOT in few-shot
    "Wie viele Schweizer Entscheidungen stammen aus dem Kanton Bern?",  # NOT in few-shot
    "What share of merits judgments after 2020 found a violation?",  # composed condition
    "How many cases involve an allegation of parental alienation?",
    "How long do proceedings take on average from application to judgment?",  # refused until 2026-07-07
    "In what proportion of cases did the mother receive custody?",
]
for q in DEMO:
    ask_anything(q)

Q: When can custody be transferred to the other parent because of alienating behaviour?
ROUTE: Bucket 1 (no aggregate trigger) -> retrieval + grounded generation
/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
  ANSWER: According to source [1], the European Court of Human Rights (ECHR) states that transfer of a child from one parent to another by means of coercion aimed at breaking the child's resistance cannot be justified if it is against the child's best interests and less restrictive measures suitable to achieve the legitimate objective in question were not seriously considered first.  Source [1] ECHR [echr:001-238568:37]  Specifically, source [1] notes that the court considers that removing the children from the alienating parent and limiting conta

## 5. What this demonstrates
- **One field is enough**: the user never declares a bucket; question *form* routes away from
  generation (measured: 0.95 accuracy), and the dispatcher decides how an aggregate is
  answered — semantically, via guarded NL→SQL, not a country lookup table.
- **Every branch is a designed behaviour**, including the refusal: Bucket-3 questions without
  a validated column are declined with the reason and the path to make them answerable, and
  every guardrail fails toward refusal, never toward a fabricated number.
- **Limits, stated**: NL→SQL is validated for *executability*, not semantic correctness — a
  syntactically valid query with the wrong intent survives the nets, which is exactly why the
  SQL prints above its result and why the citable quantitative path remains the reviewed
  canned queries in `echr_query.ipynb`. The two known router escapes (aggregate intent
  without a trigger word) apply here too.